In [62]:
import json
PDF_PATH = '../data/wheat_traits_2013.pdf'
MD_PATH = '../data/wheat_traits_2013.md'


In [135]:
# import pymupdf4llm
# md_text = pymupdf4llm.to_markdown(PDF_PATH)
md_txt = open(MD_PATH, 'r').read()
md_text_page = md_txt[98000:102000]
# print(md_text_page)

In [136]:
md_text_page

"agnostic' association of _Rht8c_ and _Xgwm261192_ applied in many Strampelli derivatives and European wheats, there was no association between reduced height and this allele in Norin 10 and its derivatives {10512}. The pedigrees of a number of Chinese wheats postulated to have _Rht8c_ on the basis of the marker trace to Italian sources {10515}. \n\n_**Rht8d**_ .   Associated with a 201-bp fragment of WMS261 {9962}. **v:** Pliska{9962}; Courtot{9962}. \n\n_**Rht8e**_ .   Associated with a 210-bp fragment of WMS261 {9962}. **v:** Chino{9962}; Klein Esterello{9962}; Klein 157{9962}. \n\n   - _**Rht8f**_ .   Associated with a 215-bp fragment of WMS261 {9962}. **v:** Klein 49{9962}. _**Rht8g**_ .   Associated with a 196-bp fragment of WMS261 [{0243}]. **v:** Mirleben{0243}. _**Rht8h**_ .   Associated with a 206-bp fragment of WMS261 [{0243}]. **v:** Weihenstephan M1{0243}. \n\n- _**Rht9**_ .   7BS{772,1601}.5AL{10249}. **v:** Acciao{718}; Forlani{718}; Mercia 12{10249}. **s:** Cappelle-Des

In [137]:
from pyparsing import (Regex, Group, Suppress, Literal, ParserElement,
                       alphanums, nums, Word, SkipTo, ZeroOrMore, Optional)

ParserElement.DEFAULT_WHITE_CHARS = ' \t'

# Trait header: **11.1. Name** with optional leading ##
trait_pat = (
    Suppress(Regex(r'#{0,2}\s*\*\*')) +
    Regex(r'(?:\d+\.)+')('index') +
    Regex(r'[^\n*]+')('name') +
    Suppress(Literal('**'))
)

# Gene id: _**GeneName**_ {optional_citation}.
gene_pat = Regex(r'_\*\*[^*\n]+\*\*_')
gene_pat.add_parse_action(lambda t: t[0][3:-3])
citation = Regex(r'\{[^}]+\}')('citation')
gene_symbol = gene_pat('gene') + Optional(citation) + Suppress(Literal('.'))

anchor = trait_pat | gene_symbol
entry = Group(
    (Group(trait_pat)('trait') | Group(gene_symbol)('gene_symbol')) +
    SkipTo(anchor)('description')
)

grammar = Suppress(SkipTo(anchor)) + ZeroOrMore(entry)

In [168]:
from pyparsing import (LineEnd, Regex, Group, 
                       Suppress, Literal, ParserElement,
                       StringEnd)

# Annotation: **label:** value until next period
BOLD = Literal('**')
gene_annot_marker = (
    Suppress(BOLD) +
    Word(alphanums)('label') +
    Suppress(Literal(':')) +
    Suppress(BOLD)
)

alt_annot = (Suppress(Literal('[')) + SkipTo(']')('alt_annot') + Suppress(Literal(']')))
chromosome_annot = (Regex(r'\d[A-Z]{1,2}')('chromosome_annot') +
                    Optional(Regex(r'\[.*\]')) + Optional(citation)
                    + Suppress(Literal('.'))
                    )
note_annot = SkipTo(gene_annot_marker | alt_annot | chromosome_annot)('note_annot')

gene_annot = gene_annot_marker + SkipTo(gene_annot_marker | 
                                        alt_annot | 
                                        chromosome_annot |
                                        # LineEnd() |
                                        StringEnd())('value')

# Lookahead anchor: where a new entry begins
next_anchor = trait_pat | gene_symbol

# Free-text description: everything before the first **label:** annotation or next entry.
# Note: chromosome annotations (e.g. "7BS{772}.") are left as part of this free text.
gene_descr = SkipTo(gene_annot | next_anchor | StringEnd())('desc')
gene_descr_annots = (Optional(note_annot) +
                Optional(alt_annot) +
                Optional(chromosome_annot) +
                ZeroOrMore(Group(gene_annot))('annots') +
                gene_descr)

In [177]:
s = "blah blah blah. 7BS. **v:** Magnif 41M1 CI 17689{718}. **ma:** Associated with 9.9 _Xwms5777B_ {10249}."
parsed_descr = gene_descr_annots.parse_string(s)
print(parsed_descr)
print("Note annotation:", parsed_descr.note_annot)
print("Alt annotation:", parsed_descr.alt_annot)
print("Chromosome annotation:", parsed_descr.chromosome_annot)
print("Gene annotations:")
for a in parsed_descr.annots:
    print(f"  {a.label}: {a.value.strip()}")

['blah blah blah.', '7BS', ['v', ' Magnif 41M1 CI 17689{718}.'], ['ma', ' Associated with 9.9 _Xwms5777B_ {10249}.'], '']
Note annotation: blah blah blah.
Alt annotation: 
Chromosome annotation: 7BS
Gene annotations:
  v: Magnif 41M1 CI 17689{718}.
  ma: Associated with 9.9 _Xwms5777B_ {10249}.


In [178]:

from gg_qa.wheat_traits.entry import (TraitEntry, GeneEntry, DocTranscription, AnnotatedGene, 
                                      Annot, NoteAnnot, ChromosomeAnnot, SynonymAnnot)

def extract_entries(text) -> DocTranscription:
    entries = []
    for e in grammar.parse_string(text):
        desc = e.description.strip()
        if 'trait' in e:
            h = e.trait
            class_name = f"{h.index.strip()} {h.name.strip()}"
            entries.append(TraitEntry(class_name=class_name, description=desc))
        else:
            g = e.gene_symbol
            # Extract annotations from description
            entries.append(GeneEntry(gene_symbol=g.gene, description=desc))
    return DocTranscription(entries=entries)

from typing import List
def extract_annotated_entries(text) -> List[TraitEntry | AnnotatedGene]:
    entries = []
    for e in grammar.parse_string(text):
        desc = e.description.strip()
        if 'trait' in e:
            h = e.trait
            class_name = f"{h.index.strip()} {h.name.strip()}"
            entries.append(TraitEntry(class_name=class_name, description=desc))
        else:
            g = e.gene_symbol[0]
            # print(e)
            # print(g)
            # print(f"Parsing gene description: {desc}")
            pds = gene_descr_annots.parse_string(desc)
            # print(f"Parsed description: {pds.dump()}")
            annots = []
            if 'note_annot' in pds:
                annots.append(Annot(label='note_annot', value=pds.note_annot))
            if 'alt_annot' in pds:
                annots.append(Annot(label='alt_annot', value=pds.alt_annot))
            if 'chromosome_annot' in pds:
                annots.append(Annot(label='chromosome_annot', value=pds.chromosome_annot))
            for a in pds.annots:
                annots.append(Annot(label=a.label, value=a.value.strip()))
            citation = pds.get('citation', '')
            trailing_desc = pds.get('desc', '')
            # Extract annotations from description
            entries.append(AnnotatedGene(gene_symbol=g, citation=citation, 
                                         annotations=annots, description=trailing_desc))
    return entries

In [179]:
s = 'Associated with a 201-bp fragment of WMS261 {9962}. **v:** Pliska{9962}; Courtot{9962}.'
parsed_descr = gene_descr_annots.parse_string(s)

In [180]:
annotated_entries = extract_annotated_entries(md_text_page)

In [181]:
annotated_entries

[AnnotatedGene(gene_symbol='Rht8d', citation='', annotations=[Annot(label='note_annot', value='Associated with a 201-bp fragment of WMS261 {9962}.'), Annot(label='v', value='Pliska{9962}; Courtot{9962}.')], description=''),
 AnnotatedGene(gene_symbol='Rht8e', citation='', annotations=[Annot(label='note_annot', value='Associated with a 210-bp fragment of WMS261 {9962}.'), Annot(label='v', value='Chino{9962}; Klein Esterello{9962}; Klein 157{9962}. \n\n   -')], description=''),
 AnnotatedGene(gene_symbol='Rht8f', citation='', annotations=[Annot(label='note_annot', value='Associated with a 215-bp fragment of WMS261 {9962}.'), Annot(label='v', value='Klein 49{9962}.')], description=''),
 AnnotatedGene(gene_symbol='Rht8g', citation='', annotations=[Annot(label='note_annot', value='Associated with a 196-bp fragment of WMS261'), Annot(label='alt_annot', value='{0243}')], description='.'),
 AnnotatedGene(gene_symbol='Rht8h', citation='', annotations=[Annot(label='note_annot', value='Associated

In [58]:
s = "- _**Rht13**_ {718}.   7BS. **v:** Magnif 41M1 CI 17689{718}. **ma:** Associated with 9.9 _Xwms5777B_ {10249}."
parsed_s = (Suppress(SkipTo(gene_symbol)) + gene_entry).parse_string(s)

In [ ]:
print(s)
print(parsed_s.dump(include_list=False))

In [ ]:


def segment(text):
    """
    Returns a flat list of dicts with 'type' in:
      'trait'            — numbered section header
      'trait_description'— unstructured text following a trait header
      'gene'             — structured gene entry: gene, citation, descr, annots
      'gene_description' — leftover text after gene_entry that wasn't parsed (rare)
    """
    segments = []
    for e in grammar.parse_string(text):
        desc = e.description.strip()
        if 'trait' in e:
            h = e.trait
            segments.append({'type': 'trait', 'index': h.index.strip(), 'name': h.name.strip()})
            key = 'trait'
        else:
            g = e.gene_entry
            seg = {
                'type':     'gene',
                'gene':     g.gene,
                'citation': g.citation or None,
                'chromosome': g.chrom or None,
                'descr':    g.descr.strip(),
                'annots':   {a.label: a.value.strip() for a in g.annots},
            }
            segments.append(seg)
            key = 'gene'
        if desc:
            segments.append({'type': f'{key}_description', 'text': desc})
    return segments


# Apply to the snippet
for seg in segment(md_text_page):
    t = seg['type']
    if t == 'trait':
        print(f"\n=== [{seg['index']}] {seg['name']}")
    elif t == 'gene':
        cit = f" {seg['citation']}" if seg['citation'] else ''
        print(f"  GENE: {seg['gene']}{cit}")
        if seg['descr']:
            print(f"    descr: {seg['descr'][:80]}")
        for label, val in seg['annots'].items():
            print(f"    {label}: {val[:80]}")
    else:
        print(f"  [{t}]: {seg['text'][:100]}")

  GENE: Rht8d
    v: Pliska{9962}; Courtot{9962}
  GENE: Rht8e
    v: Chino{9962}; Klein Esterello{9962}; Klein 157{9962}
  [gene_description]: -
  GENE: Rht8f
    v: Klein 49{9962}
  GENE: Rht8g
    v: Mirleben{0243}
  GENE: Rht8h
    v: Weihenstephan M1{0243}
  [gene_description]: -
  GENE: Rht9 {772,1601}
    v: Acciao{718}; Forlani{718}; Mercia 12{10249}
    s: Cappelle-Desprez[*] /Mara 5BS-7BS{1601}
    v2: Akakomugi _Rht8_ {1601}; Mara _Rht8_ {1601}
    ma: Close linkage with _Xwmc410-4A_ {10249}
  GENE: Rht11 {718}
    v: Karlik 1{718}
  GENE: Rht12 {718}
    bin: 5AL-23, based on co-segregation with 

_B1_ {1606}
    v: Karcagi 522M7K{721}
    ma: _Rht12_ is located distally on 5AL 

cosegregating with _B1_ and closely linked 
  [gene_description]: 4 cM - _Rht12_ {726}. 

_Rht12_ delayed ear emergence by 6 days{1606}. 

-
  GENE: Rht13 {718}
    v: Magnif 41M1 CI 17689{718}
    ma: Associated with _Xwms5777B_ {10249}
  [gene_description]: -
  GENE: Rht14 {718}
    v: Cp B 132 
